# Avocado Price Optimizer with Gurobi

## Prerequisites 
To run this notebook example you'll need to have your Nextmv API Key as a managed secret. 

```
databricks secrets put-secret --json '{
        "scope": "<scope-name>",
        "key": "nextmv-api-key",
        "string_value": "<api-key-secret>"
}
```


In [0]:
### pip install your requirments

%pip install --upgrade "nextmv[all]"
%pip install nextmv-gurobipy
%pip install gurobipy
%pip install plotly

In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
%python
api_key = dbutils.secrets.get(scope="my-scope", key="nextmv-api-key")

In [0]:
import json
import time

import nextmv
import nextmv_gurobipy as ngp
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from gurobipy import GRB
from nextmv import cloud

In [0]:
# >>>>>>>>>>>> Start Nextmv-ifying
class AvocadoPriceDecisionModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        data = input.data
        B = input.options.supply # total amount of avocado supply

        m = ngp.Model(input.options)
# <<<<<<<<<<<<< Stop Nextmv-ifying

        # Sets and parameters
        R = data["regions"]   # set of all regions

        peak_or_not = data["peak"] # 1 if it is the peak season; 1 if isn't
        year = data["year"]

        c_waste = data["cost_per_wasted_product"] # the cost ($) of wasting an avocado
        c_transport = data["transport_costs"] # the cost of transporting an avocado

        # Get the lower and upper bounds from the dataset for the price and the number of products to be stocked
        a_min = data["minimum_product_price"] # minimum avocado price in each region
        a_max = data["maximum_product_price"] # maximum avocado price in each region
        b_min = data["minimum_product_allocations"]  # minimum number of avocados allocated to each region
        b_max = data["maximum_product_allocations"]   # maximum number of avocados allocated to each region

        p = m.addVars(R,name="p",lb=a_min, ub=a_max)   # price of avocados in each region
        x = m.addVars(R,name="x",lb=b_min,ub=b_max)  # quantity supplied to each region
        s = m.addVars(R,name="s",lb=0)   # predicted amount of sales in each region for the given price
        w = m.addVars(R,name="w",lb=0)   # excess wasteage in each region

        d = {r: (data["coefficients"]['Intercept']+data["coefficients"]['price']*p[r] + data["coefficients"]['C(region)[T.%s]'%r] + data["coefficients"]['year_index']*(year-2015) + data["coefficients"]['peak']*peak_or_not) for r in R}

        m.setObjective(sum(p[r]*s[r] - c_waste*w[r] - c_transport[r]*x[r] for r in R))
        m.ModelSense = GRB.MAXIMIZE

        m.addConstr(sum(x[r] for r in R) == B)
        m.addConstrs(s[r] <= x[r] for r in R)
        m.addConstrs(s[r] <= d[r] for r in R)
        m.addConstrs(w[r] == x[r]-s[r] for r in R)
        m.Params.NonConvex = 2
        m.optimize()

        solution = pd.DataFrame()
        solution['Region'] = R
        solution['Price'] = [p[r].X for r in R]
        solution['Allocated'] = [round(x[r].X,8) for r in R]
        solution['Sold'] = [round(s[r].X,8) for r in R]
        solution['Wasted'] = [round(w[r].X,8) for r in R]
        solution['Pred_demand'] = [(data["coefficients"]['Intercept']+data["coefficients"]['price']*p[r].X + data["coefficients"]['C(region)[T.%s]'%r] + data["coefficients"]['year_index']*(year-2015) + data["coefficients"]['peak']*peak_or_not) for r in R]

        fig = px.scatter(
            solution,
            x="Price",
            y="Sold",
            color="Region",
            size="Sold",  # Size based on sold quantity
            size_max=15,  # Adjust for desired size of markers
            title="Avocado Sales and Waste by Region",
            labels={"Price": "Price per avocado ($)", "Sold": "Number of avocados sold (millions)"},
        )

        colors = px.colors.qualitative.Plotly  # Use a color palette from Plotly
        num_regions = len(solution["Region"].unique())
        region_colors = {region: colors[i % len(colors)] for i, region in enumerate(solution["Region"].unique())}


        fig.add_trace(
            go.Scatter(
                x=solution["Price"],
                y=solution["Wasted"],
                mode="markers",
                marker=dict(symbol="x", size=10, color=[region_colors[region] for region in solution["Region"]]),  # Assign colors based on region
                name="Wasted",
                showlegend=False,  # Hide legend for wasted points
            )
        )

        fig.update_layout(
            yaxis_range=[0, 5],
            xaxis_range=[1, 2.2],
            legend=dict(x=1.25, y=0.5),  # Adjust legend position
        )

        json_plot = fig.to_json()

# >>>>>>>>>>>> Start Nextmv-ifying
        statistics = ngp.ModelStatistics(m)
        statistics.result.custom = {
            "variables": m.NumVars,
            "constraints": m.NumConstrs,
            "total_waste": sum(w[r].X for r in R),
        }

        asset = nextmv.Asset(
            name="Pricing Charts",
            content_type="json",
            visual=nextmv.Visual(
                visual_schema=nextmv.VisualSchema(value="plotly"),
                label="Pricing Charts",
                visual_type="custom-tab",
            ),
            content=[json.loads(json_plot)],
        )

        return nextmv.Output(
            options=input.options,
            solution=solution.to_dict(),
            statistics=statistics,
            assets=[asset]
        )
# <<<<<<<<<<<<< Stop Nextmv-ifying


In [0]:
gp_opt = ngp.ModelOptions().to_nextmv()
nm_opt = nextmv.Options(
    nextmv.Option(name="supply", option_type=int, default=30, description="Total amount of avocado supply.", required=False),
)
options = nm_opt.merge(gp_opt)

In [0]:
## This is sample data from the avocado-ml-regressor app we'll use to test the decision model.

data = {
  "regions": [
    "Great_Lakes",
    "Midsouth",
    "Northeast",
    "Northern_New_England",
    "SouthCentral",
    "Southeast",
    "West",
    "Plains"
  ],
  "total_amount_of_supply": 30,
  "cost_per_wasted_product": 0.1,
  "peak": 1,
  "transport_costs": {
    "Great_Lakes": 0.3,
    "Midsouth": 0.1,
    "Northeast": 0.4,
    "Northern_New_England": 0.5,
    "SouthCentral": 0.3,
    "Southeast": 0.2,
    "West": 0.2,
    "Plains": 0.2
  },
  "year": 2022,
  "minimum_product_price": 0,
  "maximum_product_price": 2,
  "minimum_product_allocations": {
    "Great_Lakes": 2.06357351,
    "Midsouth": 1.8454431299999998,
    "Northeast": 2.36442449,
    "Northern_New_England": 0.2196899,
    "Plains": 1.05893809,
    "SouthCentral": 3.68713018,
    "Southeast": 2.1977637000000003,
    "West": 3.26010217
  },
  "maximum_product_allocations": {
    "Great_Lakes": 7.0947647300000005,
    "Midsouth": 6.168571610000001,
    "Northeast": 8.83640622,
    "Northern_New_England": 0.91798395,
    "Plains": 3.57549921,
    "SouthCentral": 10.32317459,
    "Southeast": 7.81047462,
    "West": 11.27474911
  },
  "coefficients": {
    "Intercept": 5.439310052165034,
    "C(region)[T.Midsouth]": -0.2426931536778823,
    "C(region)[T.Northeast]": 1.4330198366432332,
    "C(region)[T.Northern_New_England]": -3.019243790637874,
    "C(region)[T.Plains]": -1.815032872390413,
    "C(region)[T.SouthCentral]": 1.7138120739734573,
    "C(region)[T.Southeast]": 0.5839901666611691,
    "C(region)[T.West]": 2.6017994486412652,
    "price": -2.2037701048902427,
    "year_index": 0.16076930231844433,
    "peak": 0.5485105058308434,
    "C(region)[T.Great_Lakes]": 0
  }
}

In [0]:
# Run the model and check the output here in the notebook.

input = nextmv.Input(data=data, options=options)
model = AvocadoPriceDecisionModel()
output = model.solve(input)
nextmv.write_local(output)

In [0]:
### Using your api key, connect to Nextmv to push the model code as an app.

client = cloud.Client(api_key=api_key)

In [0]:
optimizer_app_name = "avocado-price-optimizer"
if cloud.Application.exists(client, id=optimizer_app_name):
    optimizer_app = cloud.Application(client=client, id=optimizer_app_name)
else:
    optimizer_app = cloud.Application.new(client=client, id=optimizer_app_name, name=optimizer_app_name)

model_configuration = nextmv.ModelConfiguration(
    name="avocado-price-optimizer",
    requirements=[
        "nextmv==0.29.0",
        "nextmv-gurobipy==0.3.0",
        "plotly==6.0.0"
    ],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)

optimizer_app.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

In [0]:
date = time.strftime("%Y-%m-%d-%H%M%S")
version = str(date)
optimizer_app.new_version(id=version, name=version)
optimizer_app.new_instance(
    version_id=version,
    id="development",
    name="development"
)
